# LLM Evaluation -- Measuring Model Quality

## Why Evaluation is Hard for Generative Models

Evaluating a classifier is straightforward: compare predictions to ground-truth labels and compute accuracy. But for generative models like LLMs, evaluation is fundamentally more challenging:

- **Many valid answers**: The question "What is the capital of France?" has one answer, but "Write a poem about the ocean" has infinitely many valid responses.
- **Quality is subjective**: Is a formal response better than a casual one? It depends on context.
- **Metrics disagree**: A response can score high on BLEU but be incoherent, or score low on BLEU but be an excellent paraphrase.
- **Training loss is misleading**: Low training loss might just mean the model memorized the training data.

Think of evaluation like grading a student:
- **Training loss** = homework score (they might be memorizing)
- **Perplexity** = multiple-choice exam (tests knowledge breadth)
- **BLEU/ROUGE** = essay grading (tests generation quality)
- **Qualitative eval** = oral examination (tests practical ability)

## What You Will Learn

1. **Perplexity** -- how "surprised" the model is by test data
2. **BLEU score** -- n-gram precision between generated and reference text
3. **ROUGE score** -- recall-oriented overlap for summarization tasks
4. **Building a simple metric from scratch** to understand the internals
5. **Automated vs. human evaluation** tradeoffs

## Prerequisites

- Familiarity with basic probability (for perplexity)
- Read the source implementation: `src/agentexplorr/llm_training/evaluation.py`

## Learning Resources

- **Perplexity explained**: [HuggingFace docs](https://huggingface.co/docs/transformers/perplexity)
- **BLEU paper**: [Papineni et al., 2002](https://aclanthology.org/P02-1040/)
- **ROUGE paper**: [Lin, 2004](https://aclanthology.org/W04-1013/)
- **Video**: [NLP Evaluation Metrics Explained](https://www.youtube.com/watch?v=TMshhnrEXlg) (Weights & Biases)
- **Blog**: [How to evaluate LLMs](https://huggingface.co/blog/evaluating-llm-models) (Hugging Face)

In [ ]:
import math
from collections import Counter

# ──────────────────────────────────────────────────────────────────────
# Imports and Setup
# ──────────────────────────────────────────────────────────────────────
# In the full evaluation pipeline (src/agentexplorr/llm_training/evaluation.py),
# we use these libraries:
#
#   import torch                        # For model inference
#   from datasets import Dataset        # For test datasets
#   from transformers import (          # For loading models
#       AutoModelForCausalLM,
#       AutoTokenizer,
#   )
#   import evaluate                     # HuggingFace metrics library
#
# For this notebook, we implement the core metrics from scratch using
# only the Python standard library + math, so you can understand
# exactly what is happening under the hood.

print("Evaluation Metrics Overview")
print("=" * 55)
print()
print("Metric         | Measures                  | Range")
print("-" * 55)
print("Perplexity     | Model surprise/confidence | 1 to inf (lower = better)")
print("BLEU           | N-gram precision          | 0 to 1  (higher = better)")
print("ROUGE-1        | Unigram recall            | 0 to 1  (higher = better)")
print("ROUGE-2        | Bigram recall             | 0 to 1  (higher = better)")
print("ROUGE-L        | Longest common subseq     | 0 to 1  (higher = better)")
print("Human Eval     | Overall quality           | Subjective (gold standard)")

## Evaluation Metrics in Detail

### 1. Perplexity (lower is better)

**"How surprised is the model by the test data?"**

Perplexity is defined as the exponential of the average cross-entropy loss:

```
Perplexity = exp(H) = exp(-1/N * sum(log P(token_i | context)))
```

If the model assigns high probability to the correct next token, perplexity is low. Intuitively, a perplexity of 10 means the model is "as confused as if it had to choose uniformly among 10 equally likely tokens at each step."

| Perplexity | Interpretation |
|------------|---------------|
| 1 | Perfect prediction (impossible in practice) |
| 5-15 | Excellent -- the model knows this domain well |
| 15-50 | Good -- reasonable performance |
| 50-100 | Mediocre -- significant uncertainty |
| >100 | Poor -- the model is very confused |

**Caveat**: Perplexity depends on the tokenizer. You cannot compare perplexity between models with different vocabularies.

### 2. BLEU Score (higher is better, 0-1)

**"How much does the generated text overlap with reference text?"**

BLEU (Bilingual Evaluation Understudy) counts matching n-grams between generated and reference text:

```
Reference: "The cat sat on the mat"
Generated: "The cat is on the mat"

Unigram matches: "The", "cat", "on", "the", "mat" --> 5/6 = 0.83
Bigram matches:  "The cat", "on the", "the mat"   --> 3/5 = 0.60
```

BLEU-4 combines 1-gram through 4-gram precision with a brevity penalty.

**Limitations**: Does not capture synonyms ("happy" vs "glad" = 0 match), semantics ("I love cats" vs "Felines are my passion" = low score), or can be gamed by repeating common words.

### 3. ROUGE Score (higher is better, 0-1)

**"How well does the generated text recall the reference content?"**

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) focuses on **recall** rather than precision:

- **ROUGE-1**: Unigram overlap (individual words)
- **ROUGE-2**: Bigram overlap (word pairs -- captures local structure)
- **ROUGE-L**: Longest Common Subsequence (captures sentence-level structure)

ROUGE is the standard metric for **summarization** tasks, while BLEU is standard for **translation**.

### 4. Qualitative Evaluation (the gold standard)

Numbers do not tell the whole story. **Always generate sample outputs and read them.** A model with perfect BLEU but incoherent responses is worse than one with lower BLEU but clear, useful answers.

Good qualitative evaluation includes:
- A **diverse set of test prompts** (simple, complex, edge cases)
- Prompts **different from training data** (tests generalization)
- Checking for **hallucination** (confident but wrong answers)
- Testing the model's ability to **say "I don't know"**

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Implementing BLEU-like Score from Scratch
# ──────────────────────────────────────────────────────────────────────
# Let's build a simplified BLEU score step by step so you can see
# exactly how n-gram matching works under the hood.

def get_ngrams(tokens, n):
    """Extract all n-grams from a list of tokens."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def simple_bleu(reference, candidate, max_n=4):
    """Compute a simplified BLEU score between a reference and candidate string.
    
    This implements the core BLEU algorithm:
      1. Compute n-gram precision for n = 1..max_n
      2. Apply brevity penalty if candidate is shorter than reference
      3. Combine with geometric mean
    """
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()
    
    # Step 1: Compute modified precision for each n-gram level
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(get_ngrams(ref_tokens, n))
        cand_ngrams = Counter(get_ngrams(cand_tokens, n))
        
        # "Modified" precision: clip each n-gram count by its reference count
        # This prevents gaming by repeating the same word many times
        clipped_count = 0
        total_count = 0
        for ngram, count in cand_ngrams.items():
            clipped_count += min(count, ref_ngrams.get(ngram, 0))
            total_count += count
        
        precision = clipped_count / total_count if total_count > 0 else 0.0
        precisions.append(precision)
    
    # Step 2: Brevity Penalty (BP)
    # Penalize candidates shorter than the reference
    if len(cand_tokens) >= len(ref_tokens):
        bp = 1.0
    elif len(cand_tokens) == 0:
        bp = 0.0
    else:
        bp = math.exp(1 - len(ref_tokens) / len(cand_tokens))
    
    # Step 3: Geometric mean of precisions (with smoothing for zero precisions)
    log_avg = 0.0
    nonzero = 0
    for p in precisions:
        if p > 0:
            log_avg += math.log(p)
            nonzero += 1
    
    if nonzero == 0:
        return {"bleu": 0.0, "precisions": precisions, "brevity_penalty": bp}
    
    bleu = bp * math.exp(log_avg / max_n)
    
    return {"bleu": bleu, "precisions": precisions, "brevity_penalty": bp}


# ──────────────────────────────────────────────────────────────────────
# Test with example sentences
# ──────────────────────────────────────────────────────────────────────
test_cases = [
    {
        "name": "Perfect match",
        "reference":  "The cat sat on the mat",
        "candidate":  "The cat sat on the mat",
    },
    {
        "name": "Close match (1 word different)",
        "reference":  "The cat sat on the mat",
        "candidate":  "The cat is on the mat",
    },
    {
        "name": "Same meaning, different words",
        "reference":  "The cat sat on the mat",
        "candidate":  "A feline rested upon the rug",
    },
    {
        "name": "Completely different",
        "reference":  "The cat sat on the mat",
        "candidate":  "I enjoy playing basketball outside",
    },
    {
        "name": "Short candidate (brevity penalty)",
        "reference":  "The cat sat on the mat",
        "candidate":  "The cat",
    },
]

print("=" * 70)
print("BLEU Score Examples (from scratch implementation)")
print("=" * 70)

for tc in test_cases:
    result = simple_bleu(tc["reference"], tc["candidate"])
    print(f"\n--- {tc['name']} ---")
    print(f"  Reference: \"{tc['reference']}\"")
    print(f"  Candidate: \"{tc['candidate']}\"")
    print(f"  BLEU Score:       {result['bleu']:.4f}")
    print(f"  Brevity Penalty:  {result['brevity_penalty']:.4f}")
    print(f"  1-gram precision: {result['precisions'][0]:.4f}")
    print(f"  2-gram precision: {result['precisions'][1]:.4f}")
    print(f"  3-gram precision: {result['precisions'][2]:.4f}")
    print(f"  4-gram precision: {result['precisions'][3]:.4f}")

## Automated vs. Human Evaluation: Tradeoffs

In practice, you need **both** automated metrics and human evaluation. Each has distinct strengths and weaknesses:

### Automated Metrics

| Pros | Cons |
|------|------|
| Fast and cheap to compute | Cannot capture semantic meaning |
| Reproducible (same inputs = same score) | Cannot detect factual errors |
| Easy to track over time | Cannot assess helpfulness or safety |
| Good for regression testing | Can be gamed (optimized without real quality) |

**Best used for**: Quick iteration during training, comparing model checkpoints, CI/CD pipelines, catching regressions.

### Human Evaluation

| Pros | Cons |
|------|------|
| Captures actual quality | Expensive and slow |
| Can assess factual correctness | Subjective (annotator disagreement) |
| Can evaluate safety/bias | Not reproducible |
| Matches real-world use | Does not scale to thousands of examples |

**Best used for**: Final quality assessment, comparing top candidates, safety review, understanding failure modes.

### LLM-as-Judge (Emerging Approach)

A growing trend is using a **stronger LLM** (e.g., GPT-4, Claude) to evaluate outputs from a weaker model. This bridges the gap between automated and human evaluation:

- Cheaper than human annotators
- More semantically aware than BLEU/ROUGE
- Can follow detailed rubrics
- But introduces its own biases (verbosity bias, position bias, self-preference)

### Recommended Evaluation Strategy

1. **During training**: Monitor training/validation loss and perplexity
2. **After training**: Compute BLEU/ROUGE on a held-out test set
3. **Before deployment**: Generate 50-100 sample outputs and review manually
4. **In production**: Collect user feedback and run periodic human evaluations

## Key Takeaways

1. **No single metric is sufficient.** Use a combination of perplexity, BLEU/ROUGE, and human evaluation to get a complete picture of model quality.

2. **Perplexity** measures how well the model predicts held-out text. It is the most reliable automated metric for language modeling but cannot capture generation quality directly.

3. **BLEU** measures n-gram **precision** (what fraction of the generated n-grams match the reference). It is best suited for translation and tasks with specific expected outputs.

4. **ROUGE** measures n-gram **recall** (what fraction of the reference n-grams appear in the output). It is the standard for summarization evaluation.

5. **All automated metrics have blind spots**: they cannot detect factual errors, assess helpfulness, or capture semantic equivalence ("happy" vs "glad"). Always supplement with qualitative review.

6. **LLM-as-Judge** is an emerging middle ground -- using a stronger model to evaluate a weaker one. It is more semantically aware than BLEU/ROUGE but cheaper than human annotators.

## Next Steps

- **Run the full evaluation pipeline**: See `src/agentexplorr/llm_training/evaluation.py` for the `ModelEvaluator` class that computes all metrics
- **Compare base vs. fine-tuned**: Load both models and run `evaluator.evaluate()` on the same test set to see the impact of fine-tuning
- **Explore the HuggingFace `evaluate` library**: [docs](https://huggingface.co/docs/evaluate) -- provides standardized implementations of BLEU, ROUGE, and many other metrics
- **Try LM Evaluation Harness**: [EleutherAI/lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) for benchmarking on standard tasks (MMLU, HellaSwag, ARC, etc.)